# Industrial Image Story Generator — 산업 이미지 스토리 생성기

산업 이미지를 입력하면 **Vision-Language Model(Qwen3-VL)** 과 **LLM(OpenAI GPT)** 을 이용하여
이미지 설명뿐만 아니라 **생산 공정**, **결함 발생 과정**, **작업 상황 스토리**까지 생성하는 교육용 PoC입니다.

| 항목 | 내용 |
|------|------|
| Vision Model | Qwen3-VL-4B-Instruct (Hugging Face Transformers) |
| LLM | OpenAI GPT (gpt-4o-mini) |
| Framework | Gradio |
| 데이터 | MVTec AD — screw, metal_nut (`MVTec_screw_metal_nut.zip`) |

### 실습 구성

| § | 내용 |
|---|------|
| 1 | 환경 설정 |
| 2 | MVTec AD 데이터셋 압축 해제 |
| 3 | Qwen3-VL 모델 로드 |
| 4 | 이미지 분석 (제품·상태·특징) |
| 5 | OpenAI 스토리 생성 |
| 6 | 통합 파이프라인 데모 |
| 7 | Gradio UI |
| 8 | 요약 |



## 1. 환경 설정

```bash
pip install -r requirements.txt
```

| 패키지 | 용도 |
|--------|------|
| `transformers>=4.57.0` | Qwen3-VL |
| `openai` | GPT 스토리 생성 |
| `gradio` | 웹 UI |
| `accelerate` | GPU 모델 로드 |

<br>

> **OpenAI API**: `C:\\env\\.env` 파일의 `OPENAI_API_KEY`가 자동 로드됩니다.

> **VRAM**: Qwen3-VL-4B 추론 기준 약 8~10GB (fp16/bf16)

<br>

**image_story.zip**을 압축 해제한 뒤 소스를 실행한다


In [1]:
# 필요 시 아래 주석 해제 후 실행
# %pip install -q "transformers>=4.57.0" accelerate pillow gradio openai

In [2]:
import sys
from pathlib import Path

import torch
from IPython.display import Markdown, display

# ── 프로젝트 루트 탐색 ──
# 노트북 실행 위치(cwd)가 프로젝트 루트가 아닐 수 있으므로
# image_story 패키지가 있는 상위 폴더를 찾아 import 경로에 추가한다.
ROOT = Path.cwd()
if not (ROOT / "image_story").exists():
    for parent in [ROOT, *ROOT.parents]:
        if (parent / "image_story").exists():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from image_story.config import STORY_TYPE_LABELS, StoryConfig
from prompt_battle.dataset import find_project_root
from prompt_battle.env_loader import load_env_file, openai_api_key_configured

# MVTec zip 파일이 있는 실제 프로젝트 루트로 재설정
ROOT = find_project_root()
# GPU 사용 가능 시 cuda, 없으면 cpu (Qwen3-VL은 GPU 권장)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
config = StoryConfig()
# C:\env\.env 에서 OPENAI_API_KEY 자동 로드
load_env_file(config.env_file)

# 환경 확인 출력
print(f"ROOT   : {ROOT}")
print(f"Device : {DEVICE}")
print(f"VLM    : {config.qwen_model_id}")
print(f"GPT    : {config.gpt_model}")
print(f"Story Types: {list(STORY_TYPE_LABELS.values())}")
print(f"OPENAI_API_KEY 설정: {openai_api_key_configured()}")

c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


ROOT   : c:\MyCursorLab\07_Vision_Language_Model
Device : cuda
VLM    : Qwen/Qwen3-VL-4B-Instruct
GPT    : gpt-4o-mini
Story Types: ['Image Description', 'Manufacturing Story', 'Defect Story', 'Incident Story', 'Prevention Guide']
OPENAI_API_KEY 설정: True


## 2. MVTec AD 데이터셋 압축 해제

`MVTec_screw_metal_nut.zip`을 압축 해제하고 screw · metal_nut test 이미지를 탐색합니다.


In [3]:
from prompt_battle.dataset import dataset_summary, list_test_images, resolve_data_root
from image_story.pipeline import ImageStoryPipeline

# 통합 파이프라인 인스턴스 생성 (이후 VLM·GPT 단계에서 재사용)
pipeline = ImageStoryPipeline(config, root=ROOT)
# MVTec_screw_metal_nut.zip 압축 해제 (이미 풀려 있으면 스킵)
pipeline.ensure_dataset()

DATA_ROOT = pipeline.data_root
# 카테고리별 결함 유형·이미지 수 집계
summary = dataset_summary(DATA_ROOT, config.categories)

print(f"DATA_ROOT: {DATA_ROOT}")
for cat, defects in summary.items():
    total = sum(defects.values())
    print(f"[{cat}] 총 {total}장 — {defects}")

# 데모용 샘플 이미지 (screw 불량 1장)
# scratch_head: 나사 머리부 스크래치 결함 유형
SAMPLE_IMAGE = list_test_images(DATA_ROOT, "screw", "scratch_head")[0]
print(f"\n샘플 이미지: {SAMPLE_IMAGE}")

DATA_ROOT: c:\MyCursorLab\07_Vision_Language_Model\MVTec_screw_metal_nut\MVTec_screw_metal_nut
[screw] 총 160장 — {'good': 41, 'manipulated_front': 24, 'scratch_head': 24, 'scratch_neck': 25, 'thread_side': 23, 'thread_top': 23}
[metal_nut] 총 115장 — {'bent': 25, 'color': 22, 'flip': 23, 'good': 22, 'scratch': 23}

샘플 이미지: c:\MyCursorLab\07_Vision_Language_Model\MVTec_screw_metal_nut\MVTec_screw_metal_nut\screw\test\scratch_head\000.png


## 3. Qwen3-VL 모델 로드

Hugging Face Transformers의 `Qwen3VLForConditionalGeneration`을 GPU에 로드합니다.


In [4]:
from PIL import Image
from prompt_battle.vlm import Qwen3VLEngine, set_shared_engine
from prompt_battle.config import BattleConfig

# StoryConfig 값을 prompt_battle의 BattleConfig로 전달
# (VLM 엔진은 prompt_battle.vlm을 재사용 — prompt_battle 소스는 수정하지 않음)
battle_cfg = BattleConfig(
    qwen_model_id=config.qwen_model_id,
    min_pixels=config.min_pixels,       # vision 토큰 하한
    max_pixels=config.max_pixels,       # vision 토큰 상한 (VRAM 절약)
    max_new_tokens=config.max_new_tokens,
    force_single_gpu=config.force_single_gpu,  # 단일 GPU 적재
)

engine = Qwen3VLEngine(battle_cfg)
engine.load()  # Hugging Face에서 가중치 다운로드 후 GPU 적재
# 노트북·Gradio가 동일 엔진을 공유하도록 전역 등록
set_shared_engine(engine)
pipeline.attach_engine(engine)

print(f"✅ Qwen3-VL 로드 완료 — device: {engine.device}, dtype: {engine.dtype}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

✅ Qwen3-VL 로드 완료 — device: cuda, dtype: torch.bfloat16


## 4. Qwen3-VL 이미지 분석

VLM이 이미지에서 다음을 추출합니다:

- **제품 종류** (product_type)
- **상태 설명** (condition)
- **특징 추출** (features)


In [5]:
from image_story.analyzer import ImageAnalyzer, ANALYSIS_PROMPT

# ImageAnalyzer: Qwen3-VL에 구조화 프롬프트를 보내 JSON 형태 분석 결과를 받는다
analyzer = ImageAnalyzer(engine, config)
sample_img = Image.open(SAMPLE_IMAGE).convert("RGB")
# analyze() → product_type, condition, features, defect_observed, summary 추출
analysis = analyzer.analyze(sample_img)

# 분석 결과를 Markdown 표로 표시
display(Markdown(analysis.to_markdown()))
print("\n--- VLM 분석 프롬프트 (교육용) ---")
print(ANALYSIS_PROMPT[:200], "...")

### Qwen3-VL 이미지 분석

| 항목 | 내용 |
|------|------|
| **제품 종류** | screw |
| **상태 설명** | 정상 |
| **관찰 결함** | 없음 |
| **요약** | 정상적인 산업용 나사로, 표면 마모는 미세한 정도이며 결함 없음 |

**특징 추출**
- 원형 평면 đầu
- 조금의 마모된 표면
- 금속색

> 분석 소요: 9.0초


--- VLM 분석 프롬프트 (교육용) ---
당신은 제조업 품질 검사 전문가입니다.
이 산업용 제품 이미지를 분석하세요.

다음 항목을 한국어로 작성하세요:
1) 제품 종류 (예: screw, metal_nut 등)
2) 상태 설명 (정상/불량 여부, 시각적 상태)
3) 특징 추출 (형태, 표면, 색상, 결함 징후 등)

반드시 아래 JSON 형식만 출력하세요:
{
  "product_type": " ...


## 5. OpenAI GPT 스토리 생성

VLM 분석 결과를 컨텍스트로 GPT가 5가지 스토리 유형 중 하나를 생성합니다.

| Story Type | 설명 |
|------------|------|
| Image Description | 이미지 내용 설명 |
| Manufacturing Story | 생산 공정 스토리 |
| Defect Story | 결함 발생 원인 추론 |
| Incident Story | 현장 상황 짧은 이야기 |
| Prevention Guide | 예방 대책 및 개선 방안 |


In [6]:
from image_story.storyteller import StoryGenerator

if not openai_api_key_configured():
    print("⚠️ OPENAI_API_KEY가 없어 이 셀은 스킵됩니다.")
else:
    # StoryGenerator: VLM 분석 결과를 컨텍스트로 OpenAI GPT에 스토리 생성 요청
    generator = StoryGenerator(config)
    story_type = "defect"  # Defect Story 데모 — 결함 발생 원인 추론
    # analysis.to_context_text()가 GPT user 프롬프트에 삽입된다
    story_text = generator.generate(analysis, story_type)
    display(Markdown(generator.format_story_output(story_type, story_text)))

## Defect Story

이 제품은 산업용 나사(screw)로, 현재 상태는 정상이며 관찰된 결함은 없습니다. 그러나 나사의 표면에 미세한 마모가 있는 점은 잠재적인 리스크로 간주될 수 있습니다. 아래는 이러한 마모가 발생할 수 있는 과정과 원인에 대한 설명입니다.

첫 번째로, 나사의 제조 공정에서 발생할 수 있는 요인입니다. 나사는 일반적으로 금속 가공 공정을 거쳐 생산됩니다. 이 과정에서 절삭, 성형, 열처리 등의 여러 단계가 포함되며, 각 단계에서 사용되는 장비의 정밀도와 상태가 품질에 영향을 미칩니다. 만약 절삭 공정에서 기계의 칼날이 마모되거나 교체 주기를 놓쳤다면, 나사의 표면이 고르지 못하게 가공되어 마모가 발생할 수 있습니다.

두 번째로, 자재의 품질이 나사의 마모에 영향을 줄 수 있습니다. 나사는 일반적으로 강철이나 알루미늄 같은 금속 재료로 제작되며, 이때 사용되는 자재의 경도와 내구성이 제품의 최종 품질에 직결됩니다. 만약 자재가 불량하거나 규격에 맞지 않는 경우, 사용 중 마찰이나 외부 충격에 의해 마모가 가속화될 수 있습니다. 

마지막으로, 제품의 취급 과정에서도 마모가 발생할 수 있는 잠재적 원인이 존재합니다. 나사가 운송 및 저장 과정에서 다른 물품과의 접촉으로 인해 긁히거나 마찰이 발생할 수 있습니다. 또한, 설치 과정에서 적절한 도구를 사용하지 않거나 과도한 힘을 가할 경우, 나사 표면에 손상이 생기고 이는 마모로 이어질 수 있습니다.

결론적으로, 현재 상태는 정상이며 결함은 없지만, 미세한 표면 마모는 제조 공정, 자재 품질, 취급 과정에서 발생할 수 있는 리스크를 나타냅니다. 이러한 잠재적 요인들을 지속적으로 모니터링하고 개선해 나가는 것이 중요합니다.

## 6. 통합 파이프라인 데모

이미지 1장 + Story Type → VLM 분석 + GPT 스토리를 한 번에 실행합니다.


In [7]:
if not openai_api_key_configured():
    print("⚠️ OPENAI_API_KEY가 없어 파이프라인 데모를 스킵합니다.")
else:
    # pipeline.run(): VLM 분석 + GPT 스토리 생성을 한 번에 실행
    result = pipeline.run(
        sample_img,
        story_type="manufacturing",  # Manufacturing Story — 생산 공정 내러티브
        image_path=SAMPLE_IMAGE,     # MVTec 경로 → ground truth 결함 유형 추출
    )
    # outputs/image_story/ 에 JSON 저장
    saved = pipeline.save_result(result)
    # VLM 분석 + GPT 스토리 + ground truth를 하나의 Markdown으로 표시
    display(Markdown(result.full_markdown()))
    print(f"\n저장: {saved}")

### Qwen3-VL 이미지 분석

| 항목 | 내용 |
|------|------|
| **제품 종류** | screw |
| **상태 설명** | 정상 |
| **관찰 결함** | 없음 |
| **요약** | 정상적인 산업용 나사로, 표면 마모는 미세한 정도이며 결함 없음 |

**특징 추출**
- 원형 평면 đầu
- 조금의 마모된 표면
- 금속색

> 분석 소요: 8.6초

---

## Manufacturing Story

### 제조 공정 이야기: 산업용 나사 제조

산업용 나사는 다양한 기계 및 구조물의 조립에 필수적인 부품입니다. 이 나사가 제조되는 과정은 여러 단계로 나뉘어 있으며, 각 단계마다 중요한 작업이 수행됩니다. 이 이야기를 통해 나사가 어떻게 만들어지는지 살펴보겠습니다.

#### 1. 원자재 준비
나사를 제조하기 위해서는 고품질의 금속 원자재가 필요합니다. 일반적으로 스테인리스강이나 탄소강이 사용됩니다. 이 단계에서는 금속 봉을 일정한 길이로 절단하여 나사의 형태를 만들기 위한 재료를 준비합니다. 예를 들어, 금속 봉이 자르는 기계에서 정확한 길이로 잘려 나가고, 이후에는 불순물을 제거하기 위해 세척 과정을 거칩니다.

#### 2. 가공 공정
원자재가 준비되면, 나사의 기본 형태를 만들기 위한 가공 단계가 시작됩니다. 이 과정에서는 CNC 선반이나 밀링 머신을 사용하여 금속 봉의 외형을 형성합니다. 예를 들어, 나사의 나선형 홈을 만들기 위해 정밀하게 가공하고, 나사 머리 부분을 다듬는 작업이 이루어집니다. 이 단계에서 나사의 형태가 완성되며, 필요한 경우 여러 차례의 가공이 반복될 수 있습니다.

#### 3. 열처리
가공이 완료된 나사는 열처리 과정을 거칩니다. 열처리는 금속의 내구성을 높이고 강도를 강화하기 위해 필수적인 단계입니다. 나사를 고온의 오븐이나 담금질 탱크에 넣어 특정 온도에서 가열한 후, 급속 냉각을 통해 금속 구조를 변형시킵니다. 이 과정은 나사의 내구성을 높이고, 나사 사용 시 발생할 수 있는 마모를 줄이는 데 중요한 역할을 합니다.

#### 4. 표면처리
열처리 후에는 표면처리 단계가 진행됩니다. 이 단계에서는 나사의 표면을 매끄럽게 하고, 부식 방지를 위해 도금이나 코팅 작업을 수행합니다. 예를 들어, 나사 표면이 깔끔하게 연마되고, 원하는 경우 아연 도금을 통해 부식에 대한 저항성을 높입니다. 이에 따라 나사는 더욱 견고하고 오래 사용할 수 있는 제품으로 변모합니다.

#### 5. 검사 및 포장
마지막으로, 제조된 나사는 품질 검사를 거칩니다. 이 과정에서는 나사의 크기, 강도, 표면 상태 등을 검토하여 결함이 없는지 확인합니다. 검사 후, 나사는 포장 단계로 넘어가며, 고객에게 안전하게 전달될 수 있도록 포장됩니다. 이 모든 과정을 통해 최종적으로 고객에게 전달되는 나사는 정밀하고, 결함이 없는 제품으로서의 역할을 다할 수 있게 됩니다.

이렇게 각 단계별로 진행되는 나사의 제조 공정은 정밀성과 품질을 보장하며, 산업 현장에서 필수적인 역할을 수행하는 제품을 만들어냅니다.

> **MVTec Ground Truth**: `scratch_head`


저장: C:\MyCursorLab\07_Vision_Language_Model\outputs\image_story\story_manufacturing_20260616_021940.json


## 7. Gradio UI

| 영역 | 구성 |
|------|------|
| **좌측** | 이미지 업로드 · MVTec 샘플 선택 · Story Type 선택 |
| **우측** | 이미지 표시 · VLM 분석 + 생성된 스토리 출력 |

```bash
python -m image_story
```


In [8]:
from image_story.gradio_app import build_demo, register_engine

# 노트북에서 로드한 VLM 엔진을 Gradio와 공유 (모델 2중 로드 방지)
register_engine(engine)
# 좌측: 이미지 업로드·Story Type / 우측: 이미지·스토리 출력
demo = build_demo(shared_engine=engine)
demo.launch(share=False)  # share=True 시 공개 URL 생성

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## 8. 요약

### 파이프라인 흐름

```
이미지 업로드
    ↓
Qwen3-VL 분석 (제품 종류 · 상태 · 특징)
    ↓
OpenAI GPT 스토리 생성 (5가지 유형)
    ↓
Gradio UI 출력
```

### 패키지 구조 (`image_story/`)

| 모듈 | 역할 |
|------|------|
| `config.py` | 모델·스토리 유형·경로 설정 |
| `analyzer.py` | Qwen3-VL 이미지 분석 |
| `storyteller.py` | OpenAI GPT 스토리 생성 |
| `pipeline.py` | 통합 파이프라인 |
| `gradio_app.py` | Gradio 웹 UI |

### 확장 아이디어

- QLoRA 파인튜닝 모델(`12`번 노트북)을 VLM 분석기로 교체
- 스토리 유형별 프롬프트 A/B 테스트
- 생성 결과를 PDF 품질 보고서로보내기
